# ResumeAI — Latih Ulang Model MLP (v2)

Notebook ini melatih ulang classifier CV **TF-IDF + MLPClassifier** dengan protokol
anti-leakage dan anti-overfit yang ketat.

## Tujuan

- Meningkatkan generalisasi dibanding model aplikasi saat ini (`mlp-tfidf-256-128-v1`).
- Tidak melakukan overfitting pada data latih.
- Tidak ada kebocoran data (data leakage) dari test ke train.
- Menghasilkan bundle model yang **kompatibel** dengan `services/ml/src/model_loader.py`.

## Protokol

1. **Bersihkan teks** dan buang baris kosong/terlalu pendek.
2. **Dedup** baris identik dan **kelompokkan near-duplicate** (cosine ≥ 0.90 pada char n-gram).
3. **Split per grup** memakai `StratifiedGroupKFold` → train 60%, validasi 20%, test 20%.
   Grup yang sama tidak pernah muncul di dua split.
4. **TF-IDF hanya di-fit di train** (di dalam `Pipeline`). Test disegel.
5. **Pemilihan hyperparameter hanya memakai validasi**, dengan penalti bila gap
   train–validasi terlalu besar.
6. **Test dievaluasi satu kali** setelah model final di-refit pada train + validasi.
7. Artefak disimpan di `latih_model/artifacts/`, model di `latih_model/model/`.

> Model aplikasi `services/ml/model/resume_analyzer_mlp.pkl` **tidak ditimpa** oleh notebook ini.

In [1]:
from __future__ import annotations

import hashlib
import html
import json
import os
import platform
import random
import re
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "package.json").exists() and (candidate / "services" / "ml").is_dir():
            return candidate
    raise RuntimeError(
        "Root proyek tidak ditemukan. Jalankan notebook dari dalam folder ResumeAI."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
NOTEBOOK_DIR = PROJECT_ROOT / "latih_model"
ARTIFACT_DIR = NOTEBOOK_DIR / "artifacts"
MODEL_DIR = NOTEBOOK_DIR / "model"
DATA_DIR = NOTEBOOK_DIR / "data"
for folder in (ARTIFACT_DIR, MODEL_DIR, DATA_DIR):
    folder.mkdir(parents=True, exist_ok=True)

APP_MODEL_PATH = PROJECT_ROOT / "services" / "ml" / "model" / "resume_analyzer_mlp.pkl"
OUTPUT_MODEL_PATH = MODEL_DIR / "resume_analyzer_mlp_v2.pkl"

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120

print("Project root :", PROJECT_ROOT)
print("Python       :", platform.python_version())
print("sklearn      :", sklearn.__version__)
print("numpy        :", np.__version__)
print("pandas       :", pd.__version__)
print("Model output :", OUTPUT_MODEL_PATH)

Project root : D:\Semester7\DL\TUGAS KELOMPOK\TES KE 2\ResumeAI
Python       : 3.11.5
sklearn      : 1.6.1
numpy        : 2.0.2
pandas       : 2.2.3
Model output : D:\Semester7\DL\TUGAS KELOMPOK\TES KE 2\ResumeAI\latih_model\model\resume_analyzer_mlp_v2.pkl


## 1. Muat dataset

Sumber utama: Kaggle `snehaanbhawal/resume-dataset`. Jika Kaggle tidak bisa diakses,
notebook memakai salinan lokal `..\Resume\Resume.csv` (dataset yang sama).

In [2]:
TEXT_COL = "Resume_str"
LABEL_COL = "Category"
KAGGLE_DATASET = "snehaanbhawal/resume-dataset"
LOCAL_CACHE = DATA_DIR / "Resume.csv"


def load_dataset() -> tuple[pd.DataFrame, str]:
    # 1) Coba Kaggle lebih dulu.
    try:
        import kagglehub

        cache_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
        csv_candidates = sorted(cache_path.rglob("Resume.csv"))
        if not csv_candidates:
            raise FileNotFoundError("Resume.csv tidak ditemukan di cache Kaggle")
        source_csv = csv_candidates[0]
        df = pd.read_csv(source_csv)
        if source_csv.resolve() != LOCAL_CACHE.resolve():
            LOCAL_CACHE.write_bytes(source_csv.read_bytes())
        print("Dataset dimuat dari Kaggle:", source_csv)
        return df, f"kaggle:{KAGGLE_DATASET}"
    except Exception as exc:
        print(f"Kaggle gagal ({type(exc).__name__}: {exc}); memakai salinan lokal.")

    # 2) Fallback lokal.
    candidates = [
        PROJECT_ROOT.parent / "Resume" / "Resume.csv",
        LOCAL_CACHE,
        PROJECT_ROOT / "Resume.csv",
    ]
    for candidate in candidates:
        if candidate.exists():
            df = pd.read_csv(candidate)
            print("Dataset dimuat dari lokal:", candidate)
            return df, f"local:{candidate}"
    raise FileNotFoundError(
        "Dataset tidak ditemukan. Unduh snehaanbhawal/resume-dataset atau letakkan "
        "Resume.csv di ..\\Resume\\Resume.csv"
    )


raw_df, DATASET_SOURCE = load_dataset()
DATASET_SHA256 = hashlib.sha256(
    (LOCAL_CACHE if LOCAL_CACHE.exists() else Path(DATASET_SOURCE.split(":", 1)[-1])).read_bytes()
).hexdigest()

missing = {TEXT_COL, LABEL_COL} - set(raw_df.columns)
if missing:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing}")

keep_cols = [c for c in ["ID", TEXT_COL, LABEL_COL] if c in raw_df.columns]
raw_df = raw_df[keep_cols].copy()
if "ID" not in raw_df.columns:
    raw_df["ID"] = np.arange(len(raw_df))

raw_df[TEXT_COL] = raw_df[TEXT_COL].fillna("").astype(str)
raw_df[LABEL_COL] = raw_df[LABEL_COL].fillna("").astype(str).str.strip()
raw_df = raw_df[raw_df[LABEL_COL] != ""].reset_index(drop=True)

print("Sumber       :", DATASET_SOURCE)
print("SHA-256      :", DATASET_SHA256)
print("Shape mentah :", raw_df.shape)
print("Jumlah kelas :", raw_df[LABEL_COL].nunique())
print("\nDistribusi kelas:")
print(raw_df[LABEL_COL].value_counts().to_string())

D:\Semester7\DL\TUGAS KELOMPOK\TES KE 2\ResumeAI\services\ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset dimuat dari Kaggle: C:\Users\ASUS\.cache\kagglehub\datasets\snehaanbhawal\resume-dataset\versions\1\Resume\Resume.csv


Sumber       : kaggle:snehaanbhawal/resume-dataset
SHA-256      : 816a7cd985a9a41e3ce8d1e83cb7e1beb1e854a89c1cf1fd0d75494faaa0559d
Shape mentah : (2484, 3)
Jumlah kelas : 24

Distribusi kelas:
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
ACCOUNTANT                118
FINANCE                   118
FITNESS                   117
AVIATION                  117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22


## 2. Pembersihan teks

Yang dibuang: tag HTML, URL, email, nomor telepon, karakter kontrol, dan spasi berlebih.
Huruf **tidak** diubah menjadi lowercase di sini karena `TfidfVectorizer` sudah menanganinya.

In [3]:
TAG_RE = re.compile(r"<[^>]+>")
URL_RE = re.compile(r"(?:https?://|www\.)\S+", flags=re.IGNORECASE)
EMAIL_RE = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b")
PHONE_RE = re.compile(r"(?<!\d)(?:\+?\d[\d\s().-]{7,}\d)(?!\d)")
SPACE_RE = re.compile(r"[ \t\f\v]+")
NEWLINE_RE = re.compile(r"\n{2,}")


def clean_resume(text: str) -> str:
    text = html.unescape(str(text))
    text = TAG_RE.sub(" ", text)
    text = URL_RE.sub(" ", text)
    text = EMAIL_RE.sub(" ", text)
    text = PHONE_RE.sub(" ", text)
    text = text.replace("\u00a0", " ")
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2013", "-").replace("\u2014", "-")
    text = SPACE_RE.sub(" ", text)
    text = NEWLINE_RE.sub("\n", text)
    return text.strip()


df = raw_df.copy()
df["text_clean"] = df[TEXT_COL].map(clean_resume)
df["word_count"] = df["text_clean"].str.split().str.len()
df["char_count"] = df["text_clean"].str.len()

before = len(df)
df = df[(df["word_count"] >= 30) & (df["char_count"] >= 200)].reset_index(drop=True)
print(f"Dibuang karena terlalu pendek : {before - len(df)} baris")

before = len(df)
df = df.drop_duplicates(subset=["text_clean"], keep="first").reset_index(drop=True)
print(f"Duplikat persis dibuang       : {before - len(df)} baris")
print(f"Data siap dipakai             : {len(df)} baris")
print(f"Kelas                         : {df[LABEL_COL].nunique()}")
print("\nStatistik panjang teks:")
print(df[["word_count", "char_count"]].describe(percentiles=[0.05, 0.5, 0.95]).round(1).to_string())

Dibuang karena terlalu pendek : 1 baris
Duplikat persis dibuang       : 2 baris
Data siap dipakai             : 2481 baris
Kelas                         : 24

Statistik panjang teks:
       word_count  char_count
count      2481.0      2481.0
mean        811.1      5923.0
std         370.6      2685.3
min         113.0       688.0
5%          302.0      2179.0
50%         756.0      5541.0
95%        1399.0     10285.0
max        5190.0     36471.0


## 3. EDA singkat

Memeriksa ketidakseimbangan kelas dan panjang CV. Kelas minoritas (`BPO`, `AUTOMOBILE`,
`AGRICULTURE`) tetap dipertahankan; metrik utama yang dipakai adalah **macro-F1**, bukan
akurasi mentah, supaya kelas kecil tidak diabaikan.

In [4]:
class_counts = df[LABEL_COL].value_counts()
imbalance_ratio = class_counts.max() / class_counts.min()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(x=class_counts.values, y=class_counts.index, ax=axes[0], color="#2563eb")
axes[0].set_title("Distribusi kelas")
axes[0].set_xlabel("Jumlah CV")
axes[0].set_ylabel("")

sns.histplot(df["word_count"], bins=50, ax=axes[1], color="#0ea5e9")
axes[1].axvline(df["word_count"].median(), color="#dc2626", linestyle="--", label="median")
axes[1].set_title("Distribusi jumlah kata per CV")
axes[1].set_xlabel("Jumlah kata")
axes[1].legend()

plt.tight_layout()
fig.savefig(ARTIFACT_DIR / "01_eda_distributions.png", dpi=140, bbox_inches="tight")
plt.show()

eda_table = pd.DataFrame(
    {
        "jumlah": class_counts,
        "persentase": (class_counts / len(df) * 100).round(2),
    }
)
print(f"Rasio kelas mayoritas/minoritas: {imbalance_ratio:.1f}x")
display(eda_table)

Rasio kelas mayoritas/minoritas: 5.5x


C:\Users\ASUS\AppData\Local\Temp\ipykernel_16844\1970289761.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,jumlah,persentase
Category,,
INFORMATION-TECHNOLOGY,120,4.84
BUSINESS-DEVELOPMENT,119,4.80
ADVOCATE,118,4.76
CHEF,118,4.76
ENGINEERING,118,4.76
ACCOUNTANT,118,4.76
FINANCE,117,4.72
FITNESS,117,4.72
SALES,116,4.68


## 4. Dedup near-duplicate → grup anti-leakage

Dua CV bisa berbeda sedikit (spasi, urutan section) tetapi isinya hampir sama. Kalau
salinannya jatuh di train dan test, skor test menjadi terlalu optimistis.

Langkah:

1. Hitung TF-IDF **character n-gram** (3–5) untuk seluruh data.
2. Cari 7 tetangga terdekat dengan cosine similarity.
3. Gabungkan pasangan dengan similarity ≥ **0.90** memakai union-find.
4. Setiap klaster menjadi satu `group_id`; split dilakukan per grup.

TF-IDF untuk clustering ini **tidak memakai label** dan tidak dipakai oleh model,
jadi bukan kebocoran. Yang penting: salinan near-duplicate tidak boleh terpisah split.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

NEAR_DUP_THRESHOLD = 0.90
t0 = time.time()

dedup_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_features=120000,
    sublinear_tf=True,
)
dedup_matrix = dedup_vectorizer.fit_transform(df["text_clean"])
print(f"Matriks char TF-IDF: {dedup_matrix.shape} ({time.time() - t0:.1f} detik)")

t0 = time.time()
n_neighbors = min(8, len(df))
neighbor_model = NearestNeighbors(
    n_neighbors=n_neighbors,
    metric="cosine",
    algorithm="brute",
).fit(dedup_matrix)
distances, neighbor_indices = neighbor_model.kneighbors(dedup_matrix)
print(f"Pencarian tetangga: {time.time() - t0:.1f} detik")

max_neighbor_similarity = 1.0 - distances[:, 1]
print(
    "Kuantil similarity tetangga terdekat:",
    np.round(np.quantile(max_neighbor_similarity, [0.5, 0.9, 0.95, 0.99, 0.999]), 4),
)

parent = np.arange(len(df))


def find_root(x: int) -> int:
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a: int, b: int) -> None:
    root_a, root_b = find_root(a), find_root(b)
    if root_a != root_b:
        parent[root_b] = root_a


joined_pairs = 0
for i in range(len(df)):
    for j, distance in zip(neighbor_indices[i, 1:], distances[i, 1:]):
        if (1.0 - distance) >= NEAR_DUP_THRESHOLD:
            union(i, int(j))
            joined_pairs += 1

df["group_id"] = [find_root(i) for i in range(len(df))]
cluster_sizes = df["group_id"].value_counts()

print(f"Pasangan near-duplicate yang digabung : {joined_pairs}")
print(f"Jumlah grup (klaster)                 : {df['group_id'].nunique()}")
print(f"Ukuran klaster terbesar               : {cluster_sizes.iloc[0]}")
print("Distribusi ukuran klaster:")
print(cluster_sizes.value_counts().sort_index().to_string())

multi_clusters = cluster_sizes[cluster_sizes > 1]
if len(multi_clusters):
    near_dup_rows = df[df["group_id"].isin(multi_clusters.index)][
        ["ID", LABEL_COL, "group_id", "word_count"]
    ].sort_values(["group_id", "ID"])
    near_dup_rows.to_csv(ARTIFACT_DIR / "near_duplicate_groups.csv", index=False)
    print(f"Detail klaster >1 disimpan: {len(near_dup_rows)} baris")
else:
    print("Tidak ada near-duplicate di atas ambang.")

Matriks char TF-IDF: (2481, 120000) (9.4 detik)


Pencarian tetangga: 14.5 detik
Kuantil similarity tetangga terdekat: [0.368  0.4648 0.5245 0.8893 0.9942]
Pasangan near-duplicate yang digabung : 26
Jumlah grup (klaster)                 : 2469
Ukuran klaster terbesar               : 3
Distribusi ukuran klaster:
count
1    2458
2      10
3       1
Detail klaster >1 disimpan: 23 baris


## 5. Split per grup: train / validasi / test

`StratifiedGroupKFold` membagi **grup**, bukan baris, sehingga salinan near-duplicate
tidak pernah berpindah split. Fold 0 menjadi test, fold 1 menjadi validasi, sisanya train.

Test **disegel**: tidak dipakai untuk memilih fitur, hyperparameter, atau ambang apa pun.

In [6]:
from sklearn.model_selection import StratifiedGroupKFold

y_all = df[LABEL_COL].astype(str)
groups_all = df["group_id"].to_numpy()

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(splitter.split(df, y_all, groups_all))

test_idx = folds[0][1]
val_idx = folds[1][1]
train_idx = np.setdiff1d(np.arange(len(df)), np.union1d(test_idx, val_idx))

assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0
assert len(train_idx) + len(val_idx) + len(test_idx) == len(df)

for split_name, indices in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    missing_classes = set(y_all.unique()) - set(y_all.iloc[indices])
    if missing_classes:
        raise AssertionError(f"Split {split_name} kehilangan kelas: {missing_classes}")

df["split"] = "train"
df.loc[val_idx, "split"] = "val"
df.loc[test_idx, "split"] = "test"

group_sets = {
    name: set(df.iloc[indices]["group_id"])
    for name, indices in [("train", train_idx), ("val", val_idx), ("test", test_idx)]
}
overlaps = {
    "train/val": len(group_sets["train"] & group_sets["val"]),
    "train/test": len(group_sets["train"] & group_sets["test"]),
    "val/test": len(group_sets["val"] & group_sets["test"]),
}
assert all(value == 0 for value in overlaps.values()), overlaps

train_texts = set(df.iloc[train_idx]["text_clean"])
test_texts = set(df.iloc[test_idx]["text_clean"])
assert train_texts.isdisjoint(test_texts)

manifest = df[["ID", LABEL_COL, "group_id", "word_count", "split"]].copy()
manifest.to_csv(ARTIFACT_DIR / "split_manifest.csv", index=False)

split_summary = (
    df.groupby("split")
    .agg(
        n_baris=("ID", "size"),
        n_grup=("group_id", "nunique"),
        n_kelas=(LABEL_COL, "nunique"),
        median_kata=("word_count", "median"),
    )
    .reindex(["train", "val", "test"])
)
print("Overlap grup antar split:", overlaps)
print(f"Overlap teks train/test : {len(train_texts & test_texts)}")
display(split_summary)

print("\nSebaran kelas per split:")
display(pd.crosstab(df[LABEL_COL], df["split"]).reindex(columns=["train", "val", "test"]))

Overlap grup antar split: {'train/val': 0, 'train/test': 0, 'val/test': 0}
Overlap teks train/test : 0


,n_baris,n_grup,n_kelas,median_kata
split,,,,
train,1487,1482,24,758.0
val,497,493,24,746.0
test,497,494,24,758.0



Sebaran kelas per split:


split,train,val,test
Category,,,
ACCOUNTANT,74,23,21
ADVOCATE,70,24,24
AGRICULTURE,39,12,12
APPAREL,56,22,19
ARTS,62,20,21
AUTOMOBILE,22,7,7
AVIATION,69,23,24
BANKING,69,24,22
BPO,12,6,4


## 6. Definisi model

Pipeline:

- `TfidfVectorizer`: unigram + bigram, `sublinear_tf=True`, `min_df=2`, `max_df=0.95`,
  maksimal 10.000 fitur, stopword Inggris.
- `MLPClassifier`: 1–2 hidden layer, ReLU, Adam, **early stopping** aktif.

Label di-encode dengan `LabelEncoder` agar early stopping sklearn 1.6.1 tidak error.
Urutan `classes_` tetap alfabetis, sama seperti bundle model aplikasi.

In [7]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    top_k_accuracy_score,
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder().fit(df[LABEL_COL])
y_encoded = label_encoder.transform(df[LABEL_COL])
CLASS_NAMES = [str(c) for c in label_encoder.classes_]
N_CLASSES = len(CLASS_NAMES)

TFIDF_PARAMS = dict(
    sublinear_tf=True,
    max_features=10000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    stop_words="english",
    token_pattern=r"(?u)\b[\w][\w.+#-]*\b",
)


def build_model(
    hidden_layer_sizes=(128,),
    alpha=1e-4,
    learning_rate_init=1e-3,
    batch_size=32,
) -> Pipeline:
    return Pipeline(
        [
            ("tfidf", TfidfVectorizer(**TFIDF_PARAMS)),
            (
                "mlp",
                MLPClassifier(
                    hidden_layer_sizes=hidden_layer_sizes,
                    activation="relu",
                    solver="adam",
                    alpha=alpha,
                    batch_size=batch_size,
                    learning_rate_init=learning_rate_init,
                    max_iter=300,
                    early_stopping=True,
                    validation_fraction=0.1,
                    n_iter_no_change=10,
                    tol=1e-4,
                    random_state=SEED,
                ),
            ),
        ]
    )


def metric_dict(model: Pipeline, X, y_true, prefix: str = "") -> dict:
    proba = model.predict_proba(X)
    pred = model.predict(X)
    return {
        f"{prefix}accuracy": float(accuracy_score(y_true, pred)),
        f"{prefix}top3_accuracy": float(
            top_k_accuracy_score(y_true, proba, k=3, labels=np.arange(N_CLASSES))
        ),
        f"{prefix}precision_macro": float(
            precision_score(y_true, pred, average="macro", zero_division=0)
        ),
        f"{prefix}recall_macro": float(
            recall_score(y_true, pred, average="macro", zero_division=0)
        ),
        f"{prefix}f1_macro": float(f1_score(y_true, pred, average="macro", zero_division=0)),
        f"{prefix}f1_weighted": float(
            f1_score(y_true, pred, average="weighted", zero_division=0)
        ),
    }


def expected_calibration_error(y_true, proba, n_bins: int = 10):
    confidence = proba.max(axis=1)
    prediction = proba.argmax(axis=1)
    correctness = (prediction == y_true).astype(float)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.clip(np.digitize(confidence, bin_edges) - 1, 0, n_bins - 1)

    rows = []
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        count = int(mask.sum())
        if count == 0:
            continue
        mean_conf = float(confidence[mask].mean())
        mean_acc = float(correctness[mask].mean())
        weight = count / len(confidence)
        ece += weight * abs(mean_acc - mean_conf)
        rows.append(
            {
                "bin": b + 1,
                "lower": float(bin_edges[b]),
                "upper": float(bin_edges[b + 1]),
                "count": count,
                "mean_confidence": mean_conf,
                "accuracy": mean_acc,
                "gap": mean_acc - mean_conf,
            }
        )
    return float(ece), pd.DataFrame(rows)


X_train = df.iloc[train_idx]["text_clean"].to_numpy()
y_train = y_encoded[train_idx]
X_val = df.iloc[val_idx]["text_clean"].to_numpy()
y_val = y_encoded[val_idx]
X_test = df.iloc[test_idx]["text_clean"].to_numpy()
y_test = y_encoded[test_idx]

print("Kelas (alfabetis):", CLASS_NAMES)
print("Train / Val / Test:", len(X_train), len(X_val), len(X_test))

Kelas (alfabetis): ['ACCOUNTANT', 'ADVOCATE', 'AGRICULTURE', 'APPAREL', 'ARTS', 'AUTOMOBILE', 'AVIATION', 'BANKING', 'BPO', 'BUSINESS-DEVELOPMENT', 'CHEF', 'CONSTRUCTION', 'CONSULTANT', 'DESIGNER', 'DIGITAL-MEDIA', 'ENGINEERING', 'FINANCE', 'FITNESS', 'HEALTHCARE', 'HR', 'INFORMATION-TECHNOLOGY', 'PUBLIC-RELATIONS', 'SALES', 'TEACHER']
Train / Val / Test: 1487 497 497


## 7. Baseline

Baseline memakai konfigurasi yang wajar: satu hidden layer 128 neuron, `alpha=1e-4`.
Angka validasi di sini adalah **out-of-sample** dan menjadi pembanding utama.

In [8]:
BASELINE_PARAMS = dict(
    hidden_layer_sizes=(128,),
    alpha=1e-4,
    learning_rate_init=1e-3,
    batch_size=32,
)

t0 = time.time()
baseline_model = build_model(**BASELINE_PARAMS)
baseline_model.fit(X_train, y_train)
baseline_seconds = time.time() - t0

baseline_train_metrics = metric_dict(baseline_model, X_train, y_train, "train_")
baseline_val_metrics = metric_dict(baseline_model, X_val, y_val, "val_")
baseline_gap = baseline_train_metrics["train_accuracy"] - baseline_val_metrics["val_accuracy"]
baseline_loss_curve = list(baseline_model.named_steps["mlp"].loss_curve_)

baseline_table = pd.DataFrame(
    {
        "train": baseline_train_metrics,
        "validation": baseline_val_metrics,
    }
)
baseline_table.index = baseline_table.index.str.replace("train_", "", regex=False).str.replace(
    "val_", "", regex=False
)
print(f"Waktu latih baseline: {baseline_seconds:.1f} detik")
print(f"Gap akurasi train-validation: {baseline_gap:.4f}")
display(baseline_table.style.format("{:.4f}"))

Waktu latih baseline: 41.9 detik
Gap akurasi train-validation: 0.2823


,train,validation
accuracy,0.9644,nan
top3_accuracy,0.9872,nan
precision_macro,0.9673,nan
recall_macro,0.9631,nan
f1_macro,0.9649,nan
f1_weighted,0.9642,nan
accuracy,nan,0.6821
top3_accuracy,nan,0.8431
precision_macro,nan,0.6844
recall_macro,nan,0.6508


## 8. Pencarian hyperparameter (hanya di train → validasi)

Kandidat diacak dari ruang kecil:

- `hidden_layer_sizes`: `(64,)`, `(128,)`, `(256,)`, `(128, 64)`, `(256, 128)`
- `alpha` (L2): `1e-4` … `1e-2`
- `learning_rate_init`: `5e-4`, `1e-3`
- `batch_size`: `16`, `32`, `64`

**Kriteria pemilihan** (bukan akurasi train):

```
selection_score = val_f1_macro - 0.25 * max(0, gap_accuracy - 0.25)
```

Artinya: utamakan macro-F1 validasi, tetapi beri penalti bila gap train–validasi
melebihi 25 poin. Dengan begitu model yang hanya menghafal train tidak menang.

In [9]:
from sklearn.model_selection import ParameterSampler

param_distributions = {
    "hidden_layer_sizes": [(64,), (128,), (256,), (128, 64), (256, 128)],
    "alpha": [1e-4, 3e-4, 1e-3, 3e-3, 1e-2],
    "learning_rate_init": [5e-4, 1e-3],
    "batch_size": [16, 32, 64],
}
N_CANDIDATES = 8
candidates = list(
    ParameterSampler(param_distributions, n_iter=N_CANDIDATES, random_state=SEED)
)

search_rows = []
best_params = None
best_score = -np.inf
best_val_metrics = None
best_train_metrics = None
best_search_loss_curve = None
best_search_n_iter = None
best_search_seconds = None

for index, params in enumerate(candidates, start=1):
    t0 = time.time()
    model = build_model(**params)
    model.fit(X_train, y_train)
    elapsed = time.time() - t0

    train_metrics = metric_dict(model, X_train, y_train, "train_")
    val_metrics = metric_dict(model, X_val, y_val, "val_")
    gap = train_metrics["train_accuracy"] - val_metrics["val_accuracy"]
    selection_score = val_metrics["val_f1_macro"] - 0.25 * max(0.0, gap - 0.25)

    search_rows.append(
        {
            "candidate": index,
            "hidden_layer_sizes": list(params["hidden_layer_sizes"]),
            "alpha": float(params["alpha"]),
            "learning_rate_init": float(params["learning_rate_init"]),
            "batch_size": int(params["batch_size"]),
            "train_accuracy": train_metrics["train_accuracy"],
            "val_accuracy": val_metrics["val_accuracy"],
            "train_f1_macro": train_metrics["train_f1_macro"],
            "val_f1_macro": val_metrics["val_f1_macro"],
            "val_f1_weighted": val_metrics["val_f1_weighted"],
            "gap_accuracy": gap,
            "selection_score": selection_score,
            "seconds": elapsed,
            "n_iter": int(model.named_steps["mlp"].n_iter_),
        }
    )

    print(
        f"[{index}/{N_CANDIDATES}] hidden={params['hidden_layer_sizes']} "
        f"alpha={params['alpha']:.0e} lr={params['learning_rate_init']:.0e} "
        f"batch={params['batch_size']} | "
        f"val_acc={val_metrics['val_accuracy']:.4f} "
        f"val_f1={val_metrics['val_f1_macro']:.4f} "
        f"gap={gap:.4f} score={selection_score:.4f} "
        f"iters={model.named_steps['mlp'].n_iter_} ({elapsed:.0f}s)"
    )

    if selection_score > best_score:
        best_score = selection_score
        best_params = dict(params)
        best_val_metrics = val_metrics
        best_train_metrics = train_metrics
        best_search_loss_curve = list(model.named_steps["mlp"].loss_curve_)
        best_search_n_iter = int(model.named_steps["mlp"].n_iter_)
        best_search_seconds = elapsed

search_df = pd.DataFrame(search_rows).sort_values("selection_score", ascending=False)
search_df = search_df.reset_index(drop=True)
search_df.to_csv(ARTIFACT_DIR / "random_search_results.csv", index=False)

print("\nPeringkat kandidat:")
display(search_df.drop(columns=["candidate"]).style.format(
    {
        "alpha": "{:.0e}",
        "learning_rate_init": "{:.0e}",
        "train_accuracy": "{:.4f}",
        "val_accuracy": "{:.4f}",
        "train_f1_macro": "{:.4f}",
        "val_f1_macro": "{:.4f}",
        "val_f1_weighted": "{:.4f}",
        "gap_accuracy": "{:.4f}",
        "selection_score": "{:.4f}",
        "seconds": "{:.1f}",
    }
))

print("\nHyperparameter terpilih:")
for key, value in best_params.items():
    print(f"  {key}: {value}")
print(f"  selection_score: {best_score:.4f}")
print(f"  val_f1_macro   : {best_val_metrics['val_f1_macro']:.4f}")
print(f"  val_accuracy   : {best_val_metrics['val_accuracy']:.4f}")
print(f"  gap_accuracy   : {best_train_metrics['train_accuracy'] - best_val_metrics['val_accuracy']:.4f}")
print(f"  n_iter         : {best_search_n_iter}")

[1/8] hidden=(128,) alpha=1e-03 lr=1e-03 batch=32 | val_acc=0.6559 val_f1=0.6060 gap=0.3024 score=0.5929 iters=18 (30s)


[2/8] hidden=(256, 128) alpha=1e-04 lr=5e-04 batch=32 | val_acc=0.6398 val_f1=0.5955 gap=0.3077 score=0.5811 iters=17 (56s)


[3/8] hidden=(256, 128) alpha=3e-03 lr=5e-04 batch=64 | val_acc=0.6640 val_f1=0.6378 gap=0.2970 score=0.6261 iters=45 (95s)


[4/8] hidden=(256, 128) alpha=1e-03 lr=5e-04 batch=32 | val_acc=0.6338 val_f1=0.5832 gap=0.2895 score=0.5733 iters=16 (66s)


[5/8] hidden=(128, 64) alpha=1e-03 lr=5e-04 batch=32 | val_acc=0.6318 val_f1=0.6112 gap=0.3332 score=0.5904 iters=32 (53s)


[6/8] hidden=(64,) alpha=3e-04 lr=1e-03 batch=16 | val_acc=0.6761 val_f1=0.6439 gap=0.2870 score=0.6347 iters=40 (66s)


[7/8] hidden=(256,) alpha=1e-03 lr=5e-04 batch=16 | val_acc=0.6881 val_f1=0.6531 gap=0.2756 score=0.6467 iters=26 (166s)


[8/8] hidden=(64,) alpha=1e-02 lr=1e-03 batch=64 | val_acc=0.6821 val_f1=0.6487 gap=0.2816 score=0.6408 iters=55 (26s)

Peringkat kandidat:


,hidden_layer_sizes,alpha,learning_rate_init,batch_size,train_accuracy,val_accuracy,train_f1_macro,val_f1_macro,val_f1_weighted,gap_accuracy,selection_score,seconds,n_iter
0,[256],1e-03,5e-04,16,0.9637,0.6881,0.9578,0.6531,0.6769,0.2756,0.6467,166.5,26
1,[64],1e-02,1e-03,64,0.9637,0.6821,0.9652,0.6487,0.6728,0.2816,0.6408,26.0,55
2,[64],3e-04,1e-03,16,0.9630,0.6761,0.9646,0.6439,0.6675,0.2870,0.6347,66.5,40
3,"[256, 128]",3e-03,5e-04,64,0.9610,0.6640,0.9581,0.6378,0.6578,0.2970,0.6261,94.7,45
4,[128],1e-03,1e-03,32,0.9583,0.6559,0.9538,0.6060,0.6401,0.3024,0.5929,30.4,18
5,"[128, 64]",1e-03,5e-04,32,0.9650,0.6318,0.9622,0.6112,0.6314,0.3332,0.5904,52.6,32
6,"[256, 128]",1e-04,5e-04,32,0.9475,0.6398,0.9173,0.5955,0.6313,0.3077,0.5811,56.3,17
7,"[256, 128]",1e-03,5e-04,32,0.9233,0.6338,0.8719,0.5832,0.6199,0.2895,0.5733,65.8,16



Hyperparameter terpilih:
  learning_rate_init: 0.0005
  hidden_layer_sizes: (256,)
  batch_size: 16
  alpha: 0.001
  selection_score: 0.6467
  val_f1_macro   : 0.6531
  val_accuracy   : 0.6881
  gap_accuracy   : 0.2756
  n_iter         : 26


## 9. Kurva pembelajaran (learning curve)

Model terbaik dilatih ulang pada sebagian data train (25%, 50%, 75%, 100%) lalu diuji
pada validasi. Tujuannya melihat apakah menambah data masih menaikkan performa, dan
seberapa besar jarak train–validasi.

Jika kurva validasi masih naik di 100%, model belum jenuh data. Jika jarak train–validasi
sangat lebar dan kurva validasi datar, model cenderung menghafal.

In [10]:
from sklearn.model_selection import train_test_split

fractions = [0.25, 0.50, 0.75, 1.00]
learning_rows = []

for fraction in fractions:
    if fraction >= 1.0:
        sub_X, sub_y = X_train, y_train
    else:
        sub_X, _, sub_y, _ = train_test_split(
            X_train,
            y_train,
            train_size=fraction,
            stratify=y_train,
            random_state=SEED,
        )

    t0 = time.time()
    model = build_model(**best_params)
    model.fit(sub_X, sub_y)
    elapsed = time.time() - t0

    pred_sub = model.predict(sub_X)
    pred_val = model.predict(X_val)
    learning_rows.append(
        {
            "fraction": fraction,
            "n_samples": len(sub_X),
            "train_accuracy": accuracy_score(sub_y, pred_sub),
            "val_accuracy": accuracy_score(y_val, pred_val),
            "train_f1_macro": f1_score(sub_y, pred_sub, average="macro", zero_division=0),
            "val_f1_macro": f1_score(y_val, pred_val, average="macro", zero_division=0),
            "seconds": elapsed,
        }
    )
    print(
        f"fraction={fraction:.2f} n={len(sub_X):4d} | "
        f"train_f1={learning_rows[-1]['train_f1_macro']:.4f} "
        f"val_f1={learning_rows[-1]['val_f1_macro']:.4f} "
        f"({elapsed:.0f}s)"
    )

learning_df = pd.DataFrame(learning_rows)
learning_df.to_csv(ARTIFACT_DIR / "learning_curve.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(learning_df["n_samples"], learning_df["train_f1_macro"], marker="o", label="Train")
ax.plot(learning_df["n_samples"], learning_df["val_f1_macro"], marker="o", label="Validasi")
ax.set_title("Kurva pembelajaran (macro-F1)")
ax.set_xlabel("Jumlah sampel train")
ax.set_ylabel("Macro-F1")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(ARTIFACT_DIR / "04_learning_curve.png", dpi=140, bbox_inches="tight")
plt.show()

display(learning_df.style.format("{:.4f}", subset=["train_accuracy", "val_accuracy", "train_f1_macro", "val_f1_macro"]))

fraction=0.25 n= 371 | train_f1=0.9486 val_f1=0.5111 (42s)


fraction=0.50 n= 743 | train_f1=0.9633 val_f1=0.6011 (78s)


fraction=0.75 n=1115 | train_f1=0.9188 val_f1=0.5853 (80s)


fraction=1.00 n=1487 | train_f1=0.9578 val_f1=0.6531 (196s)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_16844\2006314275.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,fraction,n_samples,train_accuracy,val_accuracy,train_f1_macro,val_f1_macro,seconds
0,0.250000,371,0.9488,0.5614,0.9486,0.5111,42.308892
1,0.500000,743,0.9623,0.6298,0.9633,0.6011,77.882789
2,0.750000,1115,0.9453,0.6338,0.9188,0.5853,79.939189
3,1.000000,1487,0.9637,0.6881,0.9578,0.6531,195.921560


## 10. Refit final pada train + validasi, lalu uji test **satu kali**

Setelah hyperparameter dipilih memakai validasi, model di-refit pada gabungan
train + validasi agar memanfaatkan semua data yang boleh dilihat. Test tetap disegel
sampai baris ini.

Yang dilaporkan:

- metrik pada data fit (train + validasi) → untuk melihat gap,
- metrik pada test → estimasi generalisasi,
- kalibrasi (ECE) → seberapa jujur probabilitas model.

In [11]:
X_trainval = np.concatenate([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])

t0 = time.time()
final_model = build_model(**best_params)
final_model.fit(X_trainval, y_trainval)
final_seconds = time.time() - t0

fit_metrics = metric_dict(final_model, X_trainval, y_trainval, "fit_")
test_metrics = metric_dict(final_model, X_test, y_test, "test_")
final_loss_curve = list(final_model.named_steps["mlp"].loss_curve_)
final_n_iter = int(final_model.named_steps["mlp"].n_iter_)

test_proba = final_model.predict_proba(X_test)
test_pred = final_model.predict(X_test)
test_ece, calibration_df = expected_calibration_error(y_test, test_proba, n_bins=10)
mean_confidence = float(test_proba.max(axis=1).mean())

comparison = pd.DataFrame(
    {
        "validation (out-of-sample)": best_val_metrics,
        "fit train+val": fit_metrics,
        "test (sealed)": test_metrics,
    }
)
comparison.index = (
    comparison.index.str.replace("val_", "", regex=False)
    .str.replace("fit_", "", regex=False)
    .str.replace("test_", "", regex=False)
)
print(f"Waktu refit final: {final_seconds:.1f} detik | n_iter: {final_n_iter}")
print(f"ECE test (10 bin): {test_ece:.4f} | mean confidence: {mean_confidence:.4f}")
display(comparison.style.format("{:.4f}"))

print("\nLaporan klasifikasi test:")
print(
    classification_report(
        y_test,
        test_pred,
        labels=np.arange(N_CLASSES),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )
)

Waktu refit final: 573.5 detik | n_iter: 56
ECE test (10 bin): 0.0586 | mean confidence: 0.6762


,validation (out-of-sample),fit train+val,test (sealed)
accuracy,0.6881,nan,nan
top3_accuracy,0.8491,nan,nan
precision_macro,0.6939,nan,nan
recall_macro,0.6555,nan,nan
f1_macro,0.6531,nan,nan
f1_weighted,0.6769,nan,nan
accuracy,nan,0.9733,nan
top3_accuracy,nan,0.9909,nan
precision_macro,nan,0.9746,nan
recall_macro,nan,0.9748,nan



Laporan klasifikasi test:
                        precision    recall  f1-score   support

            ACCOUNTANT     0.6000    0.7143    0.6522        21
              ADVOCATE     0.6400    0.6667    0.6531        24
           AGRICULTURE     0.5714    0.3333    0.4211        12
               APPAREL     0.8000    0.4211    0.5517        19
                  ARTS     0.5000    0.2857    0.3636        21
            AUTOMOBILE     0.2500    0.1429    0.1818         7
              AVIATION     0.6452    0.8333    0.7273        24
               BANKING     0.5909    0.5909    0.5909        22
                   BPO     0.0000    0.0000    0.0000         4
  BUSINESS-DEVELOPMENT     0.5172    0.6818    0.5882        22
                  CHEF     0.8000    0.6400    0.7111        25
          CONSTRUCTION     0.7692    0.8000    0.7843        25
            CONSULTANT     0.7059    0.5455    0.6154        22
              DESIGNER     0.8125    0.6500    0.7222        20
         DIG

## 11. Diagnostik: loss curve, confusion matrix, kalibrasi, per kelas

- **Loss curve** memperlihatkan apakah training stabil dan kapan early stopping berhenti.
- **Confusion matrix** menunjukkan kelas yang sering tertukar.
- **Reliability diagram** menunjukkan apakah confidence model cocok dengan akurasi nyata.

In [12]:
# --- Loss curves -----------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(baseline_loss_curve, label=f"Baseline (128,) — {len(baseline_loss_curve)} epoch")
ax.plot(
    best_search_loss_curve,
    label=f"Kandidat terbaik — {len(best_search_loss_curve)} epoch",
)
ax.plot(final_loss_curve, label=f"Final train+val — {len(final_loss_curve)} epoch")
ax.set_title("Loss curve (MLP, early stopping)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Log-loss")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(ARTIFACT_DIR / "07_loss_curves.png", dpi=140, bbox_inches="tight")
plt.show()

# --- Confusion matrix ------------------------------------------------------
cm = confusion_matrix(y_test, test_pred, labels=np.arange(N_CLASSES))
cm_normalized = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(16, 13))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    cbar_kws={"label": "Jumlah test"},
    ax=ax,
)
ax.set_title("Confusion matrix — test (angka = jumlah CV)")
ax.set_xlabel("Prediksi")
ax.set_ylabel("Label sebenarnya")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
fig.savefig(ARTIFACT_DIR / "05_confusion_matrix.png", dpi=140, bbox_inches="tight")
plt.show()

# --- Reliability diagram ---------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], linestyle="--", color="#64748b", label="Kalibrasi sempurna")
ax.plot(
    calibration_df["mean_confidence"],
    calibration_df["accuracy"],
    marker="o",
    color="#2563eb",
    label=f"Model (ECE={test_ece:.3f})",
)
ax.set_title("Reliability diagram — test")
ax.set_xlabel("Rata-rata confidence per bin")
ax.set_ylabel("Akurasi per bin")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(ARTIFACT_DIR / "06_reliability_diagram.png", dpi=140, bbox_inches="tight")
plt.show()

display(calibration_df.style.format({"mean_confidence": "{:.4f}", "accuracy": "{:.4f}", "gap": "{:+.4f}"}))

# --- Per-class metrics -----------------------------------------------------
report = classification_report(
    y_test,
    test_pred,
    labels=np.arange(N_CLASSES),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
per_class_df = pd.DataFrame(report).T.loc[CLASS_NAMES]
per_class_df.index.name = "category"
per_class_df = per_class_df.astype(float).sort_values("f1-score")
per_class_df.to_csv(ARTIFACT_DIR / "per_class_metrics.csv")
(ARTIFACT_DIR / "classification_report.txt").write_text(
    classification_report(
        y_test,
        test_pred,
        labels=np.arange(N_CLASSES),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    ),
    encoding="utf-8",
)

print("\nKelas dengan F1 terendah:")
display(per_class_df.head(8).style.format("{:.4f}"))
print("\nKelas dengan F1 tertinggi:")
display(per_class_df.tail(5).style.format("{:.4f}"))

C:\Users\ASUS\AppData\Local\Temp\ipykernel_16844\1026107682.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\ASUS\AppData\Local\Temp\ipykernel_16844\1026107682.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\ASUS\AppData\Local\Temp\ipykernel_16844\1026107682.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,bin,lower,upper,count,mean_confidence,accuracy,gap
0,1,0.000000,0.100000,2,0.0919,0.0000,-0.0919
1,2,0.100000,0.200000,18,0.1525,0.2222,+0.0697
2,3,0.200000,0.300000,37,0.2518,0.1892,-0.0627
3,4,0.300000,0.400000,40,0.3453,0.4750,+0.1297
4,5,0.400000,0.500000,42,0.4507,0.5952,+0.1445
5,6,0.500000,0.600000,61,0.5508,0.5246,-0.0263
6,7,0.600000,0.700000,48,0.6488,0.6042,-0.0446
7,8,0.700000,0.800000,36,0.7525,0.6944,-0.0580
8,9,0.800000,0.900000,57,0.8496,0.8246,-0.0251
9,10,0.900000,1.000000,156,0.9669,0.9231,-0.0438



Kelas dengan F1 terendah:


,precision,recall,f1-score,support
category,,,,
BPO,0.0000,0.0000,0.0000,4.0000
AUTOMOBILE,0.2500,0.1429,0.1818,7.0000
ARTS,0.5000,0.2857,0.3636,21.0000
AGRICULTURE,0.5714,0.3333,0.4211,12.0000
APPAREL,0.8000,0.4211,0.5517,19.0000
HEALTHCARE,0.5652,0.5417,0.5532,24.0000
FINANCE,0.5000,0.6364,0.5600,22.0000
BUSINESS-DEVELOPMENT,0.5172,0.6818,0.5882,22.0000



Kelas dengan F1 tertinggi:


,precision,recall,f1-score,support
category,,,,
DIGITAL-MEDIA,0.7500,0.7895,0.7692,19.0000
CONSTRUCTION,0.7692,0.8000,0.7843,25.0000
PUBLIC-RELATIONS,0.8182,0.8182,0.8182,22.0000
INFORMATION-TECHNOLOGY,0.7931,0.9583,0.8679,24.0000
HR,0.9167,0.9167,0.9167,24.0000


## 12. Simpan artefak dan bundle model

Bundle mengikuti kontrak `services/ml/src/model_loader.py`:

```python
{
    "model": Pipeline,
    "classes": list[str],          # alfabetis, sama dengan LabelEncoder.classes_
    "skill_dictionary": list[str],
    "metrics": dict,               # accuracy, top3_accuracy, precision/recall/f1 macro, f1_weighted
    "best_parameters": dict,       # key berawalan mlp__
    "random_seed": int,
}
```

Model disimpan **hanya** di `latih_model/model/resume_analyzer_mlp_v2.pkl`.

In [13]:
best_params_for_bundle = {
    "mlp__hidden_layer_sizes": list(best_params["hidden_layer_sizes"]),
    "mlp__alpha": float(best_params["alpha"]),
    "mlp__learning_rate_init": float(best_params["learning_rate_init"]),
    "mlp__batch_size": int(best_params["batch_size"]),
}

bundle_metrics = {
    key.replace("test_", "", 1): float(value) for key, value in test_metrics.items()
}

skill_dictionary = []
try:
    sys.path.insert(0, str(PROJECT_ROOT / "services" / "ml"))
    from src.skills import SKILLS

    skill_dictionary = sorted(SKILLS.keys())
except Exception as exc:
    print(f"Peringatan: gagal memuat src.skills ({type(exc).__name__}: {exc})")
    if APP_MODEL_PATH.exists():
        old_bundle = joblib.load(APP_MODEL_PATH)
        skill_dictionary = list(old_bundle.get("skill_dictionary") or [])
        print(f"Fallback skill_dictionary dari model lama: {len(skill_dictionary)} entri")

bundle = {
    "model": final_model,
    "classes": CLASS_NAMES,
    "skill_dictionary": skill_dictionary,
    "metrics": bundle_metrics,
    "best_parameters": best_params_for_bundle,
    "random_seed": SEED,
}

joblib.dump(bundle, OUTPUT_MODEL_PATH, compress=3)
print(f"Bundle disimpan: {OUTPUT_MODEL_PATH}")
print(f"Ukuran file    : {OUTPUT_MODEL_PATH.stat().st_size / 1e6:.2f} MB")

# --- Verifikasi kontrak bundle --------------------------------------------
reloaded = joblib.load(OUTPUT_MODEL_PATH)
expected_keys = {
    "model",
    "classes",
    "skill_dictionary",
    "metrics",
    "best_parameters",
    "random_seed",
}
assert set(reloaded.keys()) == expected_keys, set(reloaded.keys())
assert reloaded["classes"] == CLASS_NAMES
assert set(bundle_metrics) == {
    "accuracy",
    "top3_accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "f1_weighted",
}

sample_proba = reloaded["model"].predict_proba([X_test[0]])[0]
assert sample_proba.shape[0] == N_CLASSES
assert abs(float(sample_proba.sum()) - 1.0) < 1e-6
top3 = sorted(zip(CLASS_NAMES, sample_proba), key=lambda item: item[1], reverse=True)[:3]
print("\nSmoke test prediksi (1 CV test):")
for category, probability in top3:
    print(f"  {category:24s} {probability:.4f}")
print(f"  Label sebenarnya: {CLASS_NAMES[y_test[0]]}")

# --- Bukti kompatibel dengan loader aplikasi -------------------------------
from src.model_loader import ModelBundle

app_bundle = ModelBundle(
    pipeline=reloaded["model"],
    classes=reloaded["classes"],
    metrics=reloaded["metrics"],
    best_parameters=reloaded["best_parameters"],
    skill_dictionary=reloaded["skill_dictionary"],
    vocab_set=set(reloaded["model"].named_steps["tfidf"].vocabulary_.keys()),
)
app_ranked = app_bundle.ranked(X_test[0])[:3]
print("\nModelBundle.ranked() dari loader aplikasi:")
for item in app_ranked:
    print(f"  {item['category']:24s} {item['probability']:.4f}")
print("Kompatibel dengan model_loader.py: OK")

Bundle disimpan: D:\Semester7\DL\TUGAS KELOMPOK\TES KE 2\ResumeAI\latih_model\model\resume_analyzer_mlp_v2.pkl
Ukuran file    : 59.44 MB



Smoke test prediksi (1 CV test):
  HR                       0.8526
  PUBLIC-RELATIONS         0.0350
  AUTOMOBILE               0.0299
  Label sebenarnya: HR

ModelBundle.ranked() dari loader aplikasi:
  HR                       0.8526
  PUBLIC-RELATIONS         0.0350
  AUTOMOBILE               0.0299
Kompatibel dengan model_loader.py: OK


In [14]:
metrics_payload = {
    "model_version": "mlp-tfidf-v2",
    "trained_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": {
        "source": DATASET_SOURCE,
        "sha256": DATASET_SHA256,
        "raw_rows": int(len(raw_df)),
        "rows_after_cleaning": int(len(df)),
        "n_classes": int(N_CLASSES),
        "class_counts": {str(k): int(v) for k, v in class_counts.items()},
        "imbalance_ratio": float(imbalance_ratio),
    },
    "split": {
        "strategy": "StratifiedGroupKFold(n_splits=5) + near-duplicate groups",
        "near_duplicate_threshold": NEAR_DUP_THRESHOLD,
        "train_rows": int(len(train_idx)),
        "val_rows": int(len(val_idx)),
        "test_rows": int(len(test_idx)),
        "train_groups": int(len(group_sets["train"])),
        "val_groups": int(len(group_sets["val"])),
        "test_groups": int(len(group_sets["test"])),
        "group_overlap": overlaps,
    },
    "model": {
        "pipeline": "TfidfVectorizer -> MLPClassifier",
        "tfidf_params": TFIDF_PARAMS,
        "baseline_params": {
            "hidden_layer_sizes": list(BASELINE_PARAMS["hidden_layer_sizes"]),
            "alpha": BASELINE_PARAMS["alpha"],
            "learning_rate_init": BASELINE_PARAMS["learning_rate_init"],
            "batch_size": BASELINE_PARAMS["batch_size"],
        },
        "best_params": best_params_for_bundle,
        "search_space": {k: [list(v) if isinstance(v, tuple) else v for v in vals] for k, vals in param_distributions.items()},
        "n_candidates": N_CANDIDATES,
        "selection_score": float(best_score),
        "final_n_iter": int(final_n_iter),
    },
    "validation_out_of_sample": best_val_metrics,
    "test_sealed": test_metrics,
    "calibration_test": {
        "ece_10bin": float(test_ece),
        "mean_confidence": mean_confidence,
    },
    "runtime_seconds": {
        "baseline": float(baseline_seconds),
        "best_search_candidate": float(best_search_seconds),
        "final_refit": float(final_seconds),
        "total_search": float(search_df["seconds"].sum()),
    },
    "environment": {
        "python": platform.python_version(),
        "sklearn": sklearn.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
}

(ARTIFACT_DIR / "final_metrics.json").write_text(
    json.dumps(metrics_payload, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)
print("Artefak metrik disimpan:", ARTIFACT_DIR / "final_metrics.json")

artifact_files = sorted(p.name for p in ARTIFACT_DIR.iterdir() if p.is_file())
print("\nIsi folder artifacts:")
for name in artifact_files:
    print(" -", name)

Artefak metrik disimpan: D:\Semester7\DL\TUGAS KELOMPOK\TES KE 2\ResumeAI\latih_model\artifacts\final_metrics.json

Isi folder artifacts:
 - 01_eda_distributions.png
 - 04_learning_curve.png
 - 05_confusion_matrix.png
 - 06_reliability_diagram.png
 - 07_loss_curves.png
 - classification_report.txt
 - final_metrics.json
 - learning_curve.csv
 - near_duplicate_groups.csv
 - per_class_metrics.csv
 - random_search_results.csv
 - split_manifest.csv


In [15]:
model_card = f"""# Model Card — ResumeAI MLP v2

## Ringkasan

- **Algoritma**: TF-IDF (10.000 fitur, unigram+bigram) + MLPClassifier
- **Arsitektur**: {tuple(best_params['hidden_layer_sizes'])}
- **Regularisasi alpha**: {best_params['alpha']}
- **Batch size**: {best_params['batch_size']}
- **Learning rate**: {best_params['learning_rate_init']}
- **Early stopping**: ya (validation_fraction=0.1, n_iter_no_change=10)
- **Jumlah kelas**: {N_CLASSES}
- **Seed**: {SEED}

## Data

- Sumber: {DATASET_SOURCE}
- SHA-256: `{DATASET_SHA256}`
- Baris setelah cleaning: {len(df)}
- Split: train {len(train_idx)} / validasi {len(val_idx)} / test {len(test_idx)}
- Near-duplicate threshold: {NEAR_DUP_THRESHOLD}

## Hasil

| Metrik | Validasi (out-of-sample) | Test (sealed) |
|---|---:|---:|
| Accuracy | {best_val_metrics['val_accuracy']:.4f} | {test_metrics['test_accuracy']:.4f} |
| Top-3 accuracy | {best_val_metrics['val_top3_accuracy']:.4f} | {test_metrics['test_top3_accuracy']:.4f} |
| Precision macro | {best_val_metrics['val_precision_macro']:.4f} | {test_metrics['test_precision_macro']:.4f} |
| Recall macro | {best_val_metrics['val_recall_macro']:.4f} | {test_metrics['test_recall_macro']:.4f} |
| F1 macro | {best_val_metrics['val_f1_macro']:.4f} | {test_metrics['test_f1_macro']:.4f} |
| F1 weighted | {best_val_metrics['val_f1_weighted']:.4f} | {test_metrics['test_f1_weighted']:.4f} |

- ECE test (10 bin): {test_ece:.4f}
- Mean confidence test: {mean_confidence:.4f}
- Gap akurasi fit vs test: {fit_metrics['fit_accuracy'] - test_metrics['test_accuracy']:.4f}

## Keterbatasan

- Dataset berbahasa Inggris; CV berbahasa Indonesia akan memiliki rasio token
  in-vocabulary lebih rendah. Aplikasi sudah punya gate validasi untuk kasus ini.
- Kelas minoritas (`BPO`, `AUTOMOBILE`, `AGRICULTURE`) memiliki F1 lebih rendah.
- Probabilitas model tidak dikalibrasi ulang (ECE dilaporkan apa adanya).
- Model memakai TF-IDF, bukan embedding semantik; parafrase yang tidak berbagi
  token bisa salah klasifikasi.

## Cara memakai

Model ini **belum** menggantikan model aplikasi. Untuk promosi:

```powershell
Copy-Item "latih_model\\model\\resume_analyzer_mlp_v2.pkl" "services\\ml\\model\\resume_analyzer_mlp.pkl"
```

Lalu perbarui `MODEL_VERSION` di `services/ml/src/config.py`.
"""

(MODEL_DIR / "MODEL_CARD.md").write_text(model_card, encoding="utf-8")
print("Model card disimpan:", MODEL_DIR / "MODEL_CARD.md")

Model card disimpan: D:\Semester7\DL\TUGAS KELOMPOK\TES KE 2\ResumeAI\latih_model\model\MODEL_CARD.md


## 13. Ringkasan

Notebook selesai. Yang perlu diperiksa sebelum mempromosikan model:

1. **Test accuracy / F1 macro** dibanding model lama (accuracy 0.6273, F1 macro 0.5878).
2. **Gap fit vs test** — kalau di atas ~0.30, model masih terlalu menghafal.
3. **Per-class F1** — pastikan kelas minoritas tidak jatuh ke 0.
4. **ECE** — kalau jauh di atas 0.10, confidence model terlalu percaya diri.

Kalau hasilnya memuaskan, salin model ke aplikasi secara manual (tidak dilakukan
otomatis oleh notebook):

```powershell
Copy-Item "latih_model\model\resume_analyzer_mlp_v2.pkl" "services\ml\model\resume_analyzer_mlp.pkl"
```

Jangan lupa menaikkan `MODEL_VERSION` di `services/ml/src/config.py`, lalu restart
ML service agar model baru dimuat.

> Catatan: file `Latih model\Latih_model.ipynb` (dengan spasi) adalah file kosong
> yang tidak dipakai. Notebook aktif ada di `latih_model\Latih_model.ipynb`.